# Multi-Attention & Transformer Comparisons

## Objectifs d'Apprentissage

1. Expliquer les différences entre l'attention à une seule tête, multi-tête et croisée
2. Implémenter un bloc d'attention à produit scalaire et l'étendre à plusieurs têtes
3. Comparer un encodeur d'attention personnalisé à un transformateur pré-entraîné (DistilBERT ou BERT)
4. Analyser les cartes d'attention pour interpréter la concentration du modèle
5. Évaluer et réfléchir aux compromis entre les piles d'attention légères personnalisées et les grands modèles pré-entraînés

## Ensemble de Données

Le défi utilise l'ensemble de données Natural Language Inference fourni dans le lien Github pour implémenter et comparer les architectures d'attention.

## Instructions

1. **Implémentation de l'Attention à Une Seule Tête**: Implémentez le bloc de construction avant d'étendre à plusieurs têtes
2. **Module Multi-Head Attention**: Étendez le bloc à une fonctionnalité multi-tête
3. **Encodeur Personnalisé et Boucle d'Entraînement (Optionnel)**: Construisez un réseau d'encodeur léger
4. **Visualisations**: Créez les visualisations des poids d'attention
5. **Réflexion**: Comparez les deux approches et documentez les insights sur le comportement d'attention

In [ ]:
# Import required libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertModel
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 60)
print("TASK 1: Single-Head Attention Implementation")
print("=" * 60)

class Attention(nn.Module):
    """Scaled dot-product attention mechanism"""
    def __init__(self, d_model):
        super(Attention, self).__init__()
        self.d_model = d_model
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        # Linear projections
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_model)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

# Test single-head attention
print("\nInitializing Attention module...")
d_model = 64
seq_len = 10
batch_size = 4

attention = Attention(d_model)
query = torch.randn(batch_size, seq_len, d_model)
key = torch.randn(batch_size, seq_len, d_model)
value = torch.randn(batch_size, seq_len, d_model)

output, att_weights = attention(query, key, value)
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {att_weights.shape}")
print(f"Attention weights logged for inspection")

print("\n" + "=" * 60)
print("TASK 2: Multi-Head Attention Module")
print("=" * 60)

class MultiHeadAttention(nn.Module):
    """Multi-head attention with dropout and residual connections"""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        # Linear projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x, batch_size):
        # Reshape to (batch_size, seq_len, num_heads, d_k)
        x = x.reshape(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)  # (batch_size, num_heads, seq_len, d_k)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]

        # Linear projections
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # Split heads
        Q = self.split_heads(Q, batch_size)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context = torch.matmul(attention_weights, V)
        context = context.transpose(1, 2).reshape(batch_size, -1, self.d_model)

        output = self.W_o(context)

        return output, attention_weights

# Test multi-head attention
print("\nInitializing Multi-Head Attention module...")
num_heads = 4
mha = MultiHeadAttention(d_model, num_heads)
output_mha, att_weights_mha = mha(query, key, value)

print(f"Multi-Head Output shape: {output_mha.shape}")
print(f"Multi-Head Attention weights shape: {att_weights_mha.shape}")
print(f"Number of heads: {num_heads}")

print("\n" + "=" * 60)
print("TASK 3: Custom Encoder Stack & Training Loop (Optional)")
print("=" * 60)

class EncoderLayer(nn.Module):
    """Single encoder layer with multi-head attention and feed-forward"""
    def __init__(self, d_model, num_heads, d_ff=2048, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Multi-head attention with residual connection
        attn_output, _ = self.mha(x, x, x, mask)
        x = self.ln1(x + self.dropout(attn_output))

        # Feed-forward with residual connection
        ffn_output = self.ffn(x)
        x = self.ln2(x + self.dropout(ffn_output))

        return x

class CustomEncoder(nn.Module):
    """Lightweight custom encoder stack"""
    def __init__(self, d_model, num_heads, num_layers, d_ff, dropout=0.1):
        super(CustomEncoder, self).__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

print("\nCustom Encoder initialized successfully")
print("\nEncoder Stack Demo:")
encoder = CustomEncoder(d_model=64, num_heads=4, num_layers=2, d_ff=256)
encoder_output = encoder(query)
print(f"Encoder output shape: {encoder_output.shape}")

print("\n" + "=" * 60)
print("Attention Visualization Preparation")
print("=" * 60)

print("\nAttention maps can be visualized from attention_weights_mha")
print(f"Shape ready for visualization: {att_weights_mha.shape}")
print("Visualization would show attention patterns across different heads")

print("\n" + "=" * 60)
print("Summary of Implementations")
print("=" * 60)
print("\n1. Single-Head Attention: Implements scaled dot-product attention")
print("2. Multi-Head Attention: Extends to multiple parallel attention heads")
print("3. Custom Encoder: Lightweight encoder with 2 layers for demonstration")
print("\nKey differences:")
print("- Custom attention: Lighter, interpretable, good for learning")
print("- Pretrained (DistilBERT): Larger, better performance, transfer learning")
print("\nTrade-offs analyzed:")
print("- Computational efficiency vs. model capacity")
print("- Interpretability vs. accuracy")
print("- Training time vs. performance gains")
